# GPT-5.2 Synthetic Votes

Generates GPT-5.2 synthetic votes on the same image pairs as human votes.
Option B was selected as the final prompt following a prompt engineering evaluation
across three variants using 500 image pairs and two selection criteria:
vote distribution similarity (Jensen-Shannon divergence) and pair-level agreement
with filtered human votes (balanced accuracy, Cohen kappa).

**Toggle:** set `N_PAIRS` in section 2.
- `N_PAIRS = 100` : quick test 
- `N_PAIRS = None` :  full run (all 25,569 pairs)

**Output columns:** `user_id`, `image_name1`, `image_name2`, `vote`,
`location`, `voting_time`, `model_id`, `reason`

**Pipeline:**
1. Install dependencies
2. Imports and configuration
3. Load and deduplicate human votes
4. Azure blob connection
5. Final prompt (Option B)
6. GPT API call and vote parser
7. Run pairs
8. Export and summary

## 1. Install dependencies

In [ ]:
import sys, subprocess
for pkg in ['azure-storage-blob', 'openai', 'Pillow', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                   check=True)
print('Done.')

## 2. Imports and configuration

- Set `N_PAIRS = 100` for a quick test to verify everything works
- Set `N_PAIRS = None` for the full run (all 25,569 pairs)

In [ ]:
import os, io, re, time, base64, uuid
import pandas as pd
import numpy as np
from datetime import datetime
from PIL import Image
from tqdm import tqdm
from azure.storage.blob import BlobServiceClient
from openai import AzureOpenAI

# ============================================================
# TOGGLE — number of pairs to run
# N_PAIRS = 100     -> quick test
# N_PAIRS = None    -> full run (all unique human pairs)
# ============================================================
N_PAIRS = None

# ============================================================
# PATHS
# ============================================================
HUMAN_VOTES_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data\human_votes_05_06_26.csv'
RESULT_CSV_PATH  = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\prompt_results\gpt52_votes_25k_20_06_26.csv'

# ============================================================
# AZURE OPENAI
# ============================================================
OPENAI_MODEL_ID       = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-5.2')
AZURE_OPENAI_ENDPOINT = os.getenv('DEPLOYMENT_NAME',
                                   'https://YOUR_AZURE_OPENAI_RESOURCE.openai.azure.com/')
AZURE_OPENAI_API_KEY  = os.getenv(
    'AZURE_OPENAI_API_KEY',
    'YOUR_AZURE_OPENAI_API_KEY'
)
client = AzureOpenAI(
    api_version    = os.getenv('OPENAI_API_VERSION', '2024-12-01-preview'),
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
)

# ============================================================
# AZURE BLOB
# ============================================================
AZURE_CONN_STR = os.getenv(
    'AZURE_CONN_STR',
    'DefaultEndpointsProtocol=https;AccountName=YOUR_STORAGE_ACCOUNT;'
    'AccountKey=YOUR_AZURE_STORAGE_ACCOUNT_KEY;'
    'EndpointSuffix=core.windows.net'
)
BLOB_CONTAINER = 'london'
IMAGE_PREFIX   = 'training/training/Images/'

# ============================================================
# OUTPUT SETTINGS
# ============================================================
GPT_USER_ID      = str(uuid.uuid4())
CHECKPOINT_EVERY = 10 if (N_PAIRS is not None and N_PAIRS <= 100) else 500
RANDOM_SEED      = 42
RETRY_MAX        = 3
RATE_LIMIT_SEC   = 0.5

run_label = f'{N_PAIRS} pairs' if N_PAIRS is not None else 'full run (all pairs)'
print('Configuration loaded.')
print(f'  Model           : {OPENAI_MODEL_ID}')
print(f'  Endpoint        : {AZURE_OPENAI_ENDPOINT}')
print(f'  Run             : {run_label}')
print(f'  GPT user_id     : {GPT_USER_ID}')
print(f'  Output CSV      : {RESULT_CSV_PATH}')
print(f'  Checkpoint every: {CHECKPOINT_EVERY} results')

## 3. Load and deduplicate human votes

Deduplicate to unique pairs before sampling.
GPT votes on the same pairs as humans, one vote per unique pair.

In [ ]:
human_df = pd.read_csv(HUMAN_VOTES_PATH)

for col in ['image_name1', 'image_name2']:
    if col in human_df.columns:
        human_df[col] = human_df[col].apply(
            lambda x: os.path.basename(str(x).strip())
        )

needed  = ['image_name1', 'image_name2', 'vote']
missing = [c for c in needed if c not in human_df.columns]
if missing:
    raise ValueError(f'Missing columns in human votes: {missing}')

human_df = human_df[needed].dropna().copy()
human_df['vote'] = human_df['vote'].astype(int)
human_df['pair_key'] = human_df.apply(
    lambda r: tuple(sorted([r['image_name1'], r['image_name2']])), axis=1
)
human_df_unique = human_df.drop_duplicates(subset='pair_key').copy()

print(f'Human votes loaded     : {len(human_df):,} rows')
print(f'Unique pairs           : {len(human_df_unique):,}')
print(f'Duplicate pairs removed: {len(human_df) - len(human_df_unique)}')
print(f'Vote distribution      : '
      f'{human_df_unique["vote"].value_counts().sort_index().to_dict()}')

n_to_sample = N_PAIRS if N_PAIRS is not None else len(human_df_unique)
n_to_sample = min(n_to_sample, len(human_df_unique))

sample_df = human_df_unique.sample(
    n=n_to_sample, random_state=RANDOM_SEED
).reset_index(drop=True)

print(f'\nPairs to process        : {len(sample_df):,}')
print(f'Sample vote distribution: '
      f'{sample_df["vote"].value_counts().sort_index().to_dict()}')
print(sample_df[['image_name1', 'image_name2', 'vote']].head(5).to_string(index=False))

## 4. Azure blob connection

In [ ]:
blob_svc         = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)

def download_blob_to_pil(blob_name, timeout=30):
    blob_data = container_client.download_blob(
        blob_name, timeout=timeout
    ).readall()
    return Image.open(io.BytesIO(blob_data)).convert('RGB')

print('Testing Azure connection...')
test_blob = IMAGE_PREFIX + sample_df['image_name1'].iloc[0]
try:
    img_test = download_blob_to_pil(test_blob)
    print(f'  Connection OK: test image size: {img_test.size}')
except Exception as e:
    print(f'  Connection FAILED: {e}')
    raise

## 5. Final prompt (Option B)

Option B was selected through a prompt engineering evaluation comparing three variants
on 500 image pairs using filtered human votes as the reference.
It achieved the highest balanced accuracy (0.4357) and Cohen kappa (0.1858)
against human votes, and the lowest Jensen-Shannon divergence (0.3181) across
the three variants tested.

Design: few-shot with draw normalisation instruction embedded inline.
Three worked examples, one per response class, with content grounded in the
cycling safety perception literature (Von Stulpnagel and Binnig, 2022;
McNeil et al., 2015; Aldred and Crosweller, 2015; Branion-Calles et al., 2019).

In [ ]:
PROMPT_FINAL = (
    'You will see two images: Image A (left) and Image B (right). '
    'Which image looks safer for bicycle users? '
    'Choose 1 if Image A looks safer, 2 if Image B looks safer, '
    'or 0 if both look equally safe - many pairs look similarly safe '
    'and if there is no clear meaningful difference, choose 0.\n'
    'Respond with ONE line only:\n'
    '[VOTE]: [Vote Number] - [reason]\n\n'
    'Examples:\n'
    '[VOTE]: 1 - Image A has a physically separated cycle track and '
    'lower motor traffic volume, reducing conflict risk.\n'
    '[VOTE]: 2 - Image B has a wider road with lower motor traffic '
    'speed and no close vehicle passing.\n'
    '[VOTE]: 0 - Both streets are quiet residential roads with '
    'similar traffic volume and no dedicated cycling provision.\n\n'
)

print('Prompt loaded.')
print('=' * 60)
print(PROMPT_FINAL)
print('=' * 60)

## 6. GPT API call and vote parser

In [ ]:
def pil_to_base64_jpeg(pil_img, quality=85):
    buff = io.BytesIO()
    pil_img.save(buff, format='JPEG', quality=quality)
    return base64.b64encode(buff.getvalue()).decode('utf-8')


def call_gpt(prompt_text, pil1, pil2):
    uri1 = f'data:image/jpeg;base64,{pil_to_base64_jpeg(pil1)}'
    uri2 = f'data:image/jpeg;base64,{pil_to_base64_jpeg(pil2)}'
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'text',      'text': 'Image A:'},
            {'type': 'image_url', 'image_url': {'url': uri1}},
            {'type': 'text',      'text': '\nImage B:'},
            {'type': 'image_url', 'image_url': {'url': uri2}},
            {'type': 'text',      'text': f'\n\n{prompt_text}'},
        ],
    }]
    response = client.chat.completions.create(
        model=OPENAI_MODEL_ID,
        messages=messages,
        max_completion_tokens=256,
    )
    return response.choices[0].message.content


def parse_vote(raw_txt):
    """
    Parse GPT response into (vote_number, reason, raw_text).
    Tries three patterns in order:
    1. [VOTE]: N - reason   (expected format)
    2. N - reason           (format without tag)
    3. N                    (bare number fallback)
    Returns (None, '', raw) if no pattern matches.
    """
    raw_clean = raw_txt.strip() if raw_txt else ''
    patterns  = [
        r'^\[VOTE\]\s*[:\-]\s*([0-2])\s*[:\-]\s*(.*)$',
        r'^([0-2])\s*[:\-]\s*(.*)$',
        r'([0-2])',
    ]
    for pat in patterns:
        m = re.search(pat, raw_clean, re.IGNORECASE | re.MULTILINE)
        if m:
            vote_num = int(m.group(1))
            reason   = m.group(2).strip() if len(m.groups()) >= 2 else ''
            return vote_num, reason, raw_clean
    return None, '', raw_clean


print('API client and parser ready.')

## 7. Run pairs

Output columns: `user_id`, `image_name1`, `image_name2`, `vote`, `location`,
`voting_time`, `model_id`, `reason`.

Image names include the full path prefix `training/training/Images/`.

Content policy violations are skipped immediately without retrying.
Checkpoints save to disk every `CHECKPOINT_EVERY` successful results.

In [ ]:
results        = []
parse_failures = []
skipped        = []
total_pairs    = len(sample_df)

print(f'Running {total_pairs:,} pairs with {OPENAI_MODEL_ID}...')
print(f'GPT user_id : {GPT_USER_ID}')
print('=' * 70)

for i, row in tqdm(sample_df.iterrows(), total=total_pairs, desc='Pairs'):
    name1 = row['image_name1']
    name2 = row['image_name2']

    # ── download images with retry ────────────────────────────────────────
    pil1 = pil2 = None
    for attempt in range(RETRY_MAX):
        try:
            pil1 = download_blob_to_pil(IMAGE_PREFIX + name1)
            pil2 = download_blob_to_pil(IMAGE_PREFIX + name2)
            break
        except Exception as e:
            if attempt == RETRY_MAX - 1:
                print(f'  SKIP (download) {name1} / {name2}: {e}')
                skipped.append({
                    'image_name1': IMAGE_PREFIX + name1,
                    'image_name2': IMAGE_PREFIX + name2,
                    'reason'     : f'download_failed: {str(e)[:80]}'
                })
            time.sleep(2 ** attempt)

    if pil1 is None or pil2 is None:
        continue

    # ── call GPT with retry ───────────────────────────────────────────────
    raw_txt   = None
    vote_time = datetime.now().strftime('%d/%m/%Y %H:%M')
    for attempt in range(RETRY_MAX):
        try:
            raw_txt   = call_gpt(PROMPT_FINAL, pil1, pil2)
            vote_time = datetime.now().strftime('%d/%m/%Y %H:%M')
            break
        except Exception as e:
            err_str = str(e)
            if 'content_policy_violation' in err_str or '400' in err_str:
                print(f'  SKIP (content policy) {name1} / {name2}')
                skipped.append({
                    'image_name1': IMAGE_PREFIX + name1,
                    'image_name2': IMAGE_PREFIX + name2,
                    'reason'     : 'content_policy_violation'
                })
                break
            if attempt == RETRY_MAX - 1:
                print(f'  SKIP (API) {name1} / {name2}: {err_str[:80]}')
                skipped.append({
                    'image_name1': IMAGE_PREFIX + name1,
                    'image_name2': IMAGE_PREFIX + name2,
                    'reason'     : f'api_failed: {err_str[:80]}'
                })
            time.sleep(2 ** attempt)

    if raw_txt is None:
        continue

    # ── parse response ────────────────────────────────────────────────────
    ai_vote, reason, raw_clean = parse_vote(raw_txt)

    if ai_vote is None:
        print(f'  PARSE FAIL {name1} / {name2}: {raw_clean[:80]}')
        parse_failures.append({
            'image_name1': IMAGE_PREFIX + name1,
            'image_name2': IMAGE_PREFIX + name2,
            'raw_txt'    : raw_clean
        })
        skipped.append({
            'image_name1': IMAGE_PREFIX + name1,
            'image_name2': IMAGE_PREFIX + name2,
            'reason'     : f'parse_failed: {raw_clean[:80]}'
        })
        continue

    # ── clean encoding ────────────────────────────────────────────────────
    reason = reason.encode('utf-8', errors='replace').decode('utf-8')

    # ── save result ───────────────────────────────────────────────────────
    results.append({
        'user_id'     : GPT_USER_ID,
        'image_name1' : IMAGE_PREFIX + name1,
        'image_name2' : IMAGE_PREFIX + name2,
        'vote'        : ai_vote,
        'location'    : '',
        'voting_time' : vote_time,
        'model_id'    : OPENAI_MODEL_ID,
        'reason'      : reason,
    })

    # ── progress every 10 results ─────────────────────────────────────────
    if len(results) % 10 == 0:
        pct  = len(results) / total_pairs * 100
        dr   = sum(1 for r in results if r['vote'] == 0) / len(results) * 100
        print(
            f'  [{len(results):>5} / {total_pairs:,}]  '
            f'{pct:>5.1f}%  '
            f'draw={dr:.1f}%  '
            f'skipped={len(skipped)}  '
            f'failures={len(parse_failures)}'
        )

    # ── checkpoint ───────────────────────────────────────────────────────
    if len(results) % CHECKPOINT_EVERY == 0 and len(results) > 0:
        pd.DataFrame(results).to_csv(
            RESULT_CSV_PATH, index=False, encoding='utf-8-sig'
        )
        print(f'  Checkpoint saved: {len(results):,} results')

    time.sleep(RATE_LIMIT_SEC)

print('\n' + '=' * 70)
print('Run complete.')
print(f'  Processed      : {len(results):,}')
print(f'  Parse failures : {len(parse_failures)}')
print(f'  Skipped        : {len(skipped)}')

## 8. Export and summary

In [ ]:
if not results:
    print('No results to save. Check the error messages above.')
else:
    final_df = pd.DataFrame(results)
    final_df.to_csv(RESULT_CSV_PATH, index=False, encoding='utf-8-sig')

    n         = len(final_df)
    draw_rate = (final_df['vote'] == 0).mean()
    vote_dist = final_df['vote'].value_counts(normalize=True).sort_index()

    print('=' * 60)
    print('SUMMARY')
    print('=' * 60)
    print(f'  Saved to       : {RESULT_CSV_PATH}')
    print(f'  Total pairs    : {n:,}')
    print(f'  Parse failures : {len(parse_failures)}')
    print(f'  Skipped        : {len(skipped)}')
    print(f'  GPT draw rate  : {draw_rate*100:.2f}%')
    print()
    print('  Vote distribution:')
    for v, pct in vote_dist.items():
        bar = '#' * int(pct * 30)
        print(f'    {int(v)} : {bar:<30} {pct*100:.1f}%')
    print()
    print('  Output columns:')
    print(f'    {list(final_df.columns)}')
    print()
    print('  Sample output (first 3 rows):')
    print(final_df[['user_id','image_name1','vote',
                    'voting_time','model_id','reason']]
          .head(3).to_string(index=False))
    print()
    print(f'  Recommended DRAW_PROB_VLM for Stage 1 TrueSkill: {draw_rate:.4f}')

    if parse_failures:
        print(f'\nParse failures ({len(parse_failures)}):')
        for pf in parse_failures:
            print(f"  {pf['image_name1']} / {pf['image_name2']}")
            print(f"    raw: {pf['raw_txt'][:100]}")

    if skipped:
        print(f'\nSkipped pairs ({len(skipped)}):')
        from collections import Counter
        reasons = Counter(s['reason'].split(':')[0] for s in skipped)
        for reason, count in reasons.most_common():
            print(f'  {reason}: {count}')